In [27]:
!ollama serve 
!pip install requests
!ollama run deepseek-r1:7b

Error: listen tcp 127.0.0.1:11434: bind: address already in use
⠙ ⠹ ⠸ ⠸ ⠴ ⠴ >>> Send a message (/? for help)

OSError: [Errno 5] Input/output error

In [28]:
import requests

response = requests.post(
    "http://localhost:11434/api/chat",
    json={
        "model": "deepseek-r1:7b",  # make sure this model is pulled via `ollama pull deepseek:7b`
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "What is the capital of California?"}
        ]
    }
)

print(response.status_code)
print(response.text)

200
{"model":"deepseek-r1:7b","created_at":"2025-04-14T23:34:05.458329Z","message":{"role":"assistant","content":"\u003cthink\u003e"},"done":false}
{"model":"deepseek-r1:7b","created_at":"2025-04-14T23:34:05.487231Z","message":{"role":"assistant","content":"\n\n"},"done":false}
{"model":"deepseek-r1:7b","created_at":"2025-04-14T23:34:05.524852Z","message":{"role":"assistant","content":"\u003c/think\u003e"},"done":false}
{"model":"deepseek-r1:7b","created_at":"2025-04-14T23:34:05.562555Z","message":{"role":"assistant","content":"\n\n"},"done":false}
{"model":"deepseek-r1:7b","created_at":"2025-04-14T23:34:05.600179Z","message":{"role":"assistant","content":"The"},"done":false}
{"model":"deepseek-r1:7b","created_at":"2025-04-14T23:34:05.637252Z","message":{"role":"assistant","content":" capital"},"done":false}
{"model":"deepseek-r1:7b","created_at":"2025-04-14T23:34:05.676083Z","message":{"role":"assistant","content":" of"},"done":false}
{"model":"deepseek-r1:7b","created_at":"2025-04-14

In [ ]:
# ollama_chatbot.ipynb

import requests
import json

# 🔧 Set your model and system prompt
OLLAMA_MODEL = "<INSERT YOUR MODEL NAME HERE"
OLLAMA_API = "http://localhost:11434/api/chat"

system_prompt = """
You are a data insights assistant for a nonprofit car sharing service called MioCar.
You analyze booking logs in a compressed format. Each line represents a booking:
[Status Initial] Station | Booking Creation time, Pickup time → Dropoff time | Vehicle Make Model | Plate | Total Revenue | Trip Distance

Legend:
- "C" = Cancelled, "F" = Finished
- Times are in MM/DD HH:MM format
- Trip Distance is in miles
- Canceled bookings reflect a charge of $0

Your goal is to extract insights useful for MioCar admins, including but not limited to:
- Spikes or drops in activity at specific stations
- Gaps in usage or inconsistent booking patterns
- Interesting revenue insights throughout the period of time

Ignore unimportant trivia. Focus only on patterns relevant to MioCar staff overseeing fleet usage and customer behavior and outlook.

Your output should begin with a brief, business-focused paragraph summary of key findings. Follow this with actionable bullet points for any items that require attention, optimization, or further investigation.
 Do not include <think> tags. Respond directly to the user's request in your final answer only.
"""

# 🧠 Function to chat with the model
def ask_ollama(message, history=[]):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = requests.post(OLLAMA_API, json= {
                                                "model": OLLAMA_MODEL,
                                                "messages": messages,
                                                "stream": False
                                                }
                            )
    response.raise_for_status()
    print(response.status_code)
    print(response.text)

# Example booking log input (Example) 
booking_data = """
C  | Station A | 01/01 08:00, 01/01 09:00 → 01/01 10:00 | Toyota Prius | ABC123 | $0 | 0
F  | Station B | 01/01 10:00, 01/01 11:00 → 01/01 12:00 | Honda Civic | XYZ789 | $20 | 10
F  | Station A | 01/01 12:00, 01/01 13:00 → 01/01 14:00 | Ford Focus | DEF456 | $15 | 8
"""

# 💬 Call the assistant
ask_ollama(booking_data)


200
{"model":"deepseek-r1:7b","created_at":"2025-04-15T01:27:55.690986Z","message":{"role":"assistant","content":"\u003cthink\u003e\nOkay, so I've got this data set from MioCar's booking logs. The goal is to figure out how to help the admins understand their operations better by analyzing these logs. Let me start by understanding what each line represents.\n\nEach line has a status, station name, times for creation and pickup to dropoff, vehicle info, plate, revenue, and trip distance. Status can be 'C' for cancelled or 'F' for finished. The first thing I notice is that all the cancellations have $0 revenue, so they don't contribute much to income.\n\nLooking at the data, there are three bookings on 01/01. Both Station A and Station B have some finished bookings. Station B has one booking with a vehicle and plate of XYZ789, but it's finished, so I wonder if that's completed or maybe it was a mistake? The revenue is $20 for that one.\n\nFor Station A, there are two more finished booking